# 02 — Transform raw data into a state analytical panel

This is the **transform** stage. It reads the five raw files saved by notebook `01`, cleans and standardises each one, and merges them into a single tidy table — **one row per state, one column per variable** — written to:

```text
../data/processed/state_religiosity_outcomes.csv
```

This is the file every analysis in notebook `03` reads.

### The core problem this notebook solves

Each raw source labels and formats states differently, carries extra rows (national totals, DC, territories), and stores numbers as messy strings (`"1,234"`, `"38.9%"`, `"NA"`). Merging them naively would silently drop or mismatch states. So the whole notebook is built around two guarantees:

1. **Canonical states** — every source is mapped to the same 50 official state names before joining, so merges are exact rather than fuzzy.
2. **Fail loud, not silent** — helper functions validate columns, numeric ranges, and the 50-state count, raising an error the moment something is off instead of producing a quietly wrong panel.

We also attach each state's **Census region** (`Northeast / Midwest / South / West`), which notebook `03` uses to test whether any religiosity–outcome association is just a regional pattern.

In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

STATES: dict[str, str] = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_TO_ABBR = {state: abbr for abbr, state in STATES.items()}
EXPECTED_STATES = set(STATES.values())
EXPECTED_STATE_ABBRS = set(STATES.keys())

CENSUS_REGION: dict[str, str] = {
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast", "New Hampshire": "Northeast",
    "Rhode Island": "Northeast", "Vermont": "Northeast", "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    "Illinois": "Midwest", "Indiana": "Midwest", "Michigan": "Midwest", "Ohio": "Midwest", "Wisconsin": "Midwest",
    "Iowa": "Midwest", "Kansas": "Midwest", "Minnesota": "Midwest", "Missouri": "Midwest", "Nebraska": "Midwest", "North Dakota": "Midwest", "South Dakota": "Midwest",
    "Delaware": "South", "Florida": "South", "Georgia": "South", "Maryland": "South", "North Carolina": "South", "South Carolina": "South", "Virginia": "South", "West Virginia": "South",
    "Alabama": "South", "Kentucky": "South", "Mississippi": "South", "Tennessee": "South",
    "Arkansas": "South", "Louisiana": "South", "Oklahoma": "South", "Texas": "South",
    "Arizona": "West", "Colorado": "West", "Idaho": "West", "Montana": "West", "Nevada": "West", "New Mexico": "West", "Utah": "West", "Wyoming": "West",
    "Alaska": "West", "California": "West", "Hawaii": "West", "Oregon": "West", "Washington": "West",
}

assert len(CENSUS_REGION) == 50
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Raw dir: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\raw
Processed dir: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\processed


## Helper functions

These utilities do the heavy lifting of cleaning and validation, so each source section below stays short. The key ones:

- **`normalise_state_name`** — accepts a full name, an abbreviation, mixed casing, or a name with trailing junk, and returns the canonical state name (or `None` for non-states like "District of Columbia" or "United States"). This is what makes the later merges exact.
- **`to_float`** — robustly converts messy strings to floats, treating `"NA"`, `"-"`, `"n/a"`, etc. as missing rather than erroring.
- **`require_columns`** — asserts the expected columns exist, with a helpful message listing what is actually present.
- **`coerce_numeric_series`** — parses a column to numbers *and* range-checks it (e.g. a percentage must be 0–100), raising with a sample of offending values if anything looks wrong. This catches schema drift early.
- **`require_50_states`** — the gatekeeper. It maps the source's state column to canonical names, drops non-state rows, and then **asserts the result is exactly the 50 states with no duplicates and none missing**. If a source unexpectedly loses or gains a state, the notebook stops here.

As in notebook `01`, imports and the state/region dictionaries are repeated so this cell is self-contained after a kernel restart.

In [2]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

STATES: dict[str, str] = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_TO_ABBR = {state: abbr for abbr, state in STATES.items()}
ABBR_TO_STATE = {abbr: state for abbr, state in STATES.items()}
EXPECTED_STATES = set(STATES.values())
EXPECTED_STATE_ABBRS = set(STATES.keys())

NON_STATE_LABELS = {
    "USA", "US", "UNITED STATES", "TOTAL", "NATIONAL", "UNITED STATES OF AMERICA",
    "DISTRICT OF COLUMBIA", "DC", "D.C.", "PUERTO RICO", "GUAM", "AMERICAN SAMOA", "COMMONWEALTH",
    "NORTHERN MARIANA ISLANDS", "VIRGIN ISLANDS",
}

CENSUS_REGION: dict[str, str] = {
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast", "New Hampshire": "Northeast",
    "Rhode Island": "Northeast", "Vermont": "Northeast", "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    "Illinois": "Midwest", "Indiana": "Midwest", "Michigan": "Midwest", "Ohio": "Midwest", "Wisconsin": "Midwest",
    "Iowa": "Midwest", "Kansas": "Midwest", "Minnesota": "Midwest", "Missouri": "Midwest", "Nebraska": "Midwest", "North Dakota": "Midwest", "South Dakota": "Midwest",
    "Delaware": "South", "Florida": "South", "Georgia": "South", "Maryland": "South", "North Carolina": "South", "South Carolina": "South", "Virginia": "South", "West Virginia": "South",
    "Alabama": "South", "Kentucky": "South", "Mississippi": "South", "Tennessee": "South",
    "Arkansas": "South", "Louisiana": "South", "Oklahoma": "South", "Texas": "South",
    "Arizona": "West", "Colorado": "West", "Idaho": "West", "Montana": "West", "Nevada": "West", "New Mexico": "West", "Utah": "West", "Wyoming": "West",
    "Alaska": "West", "California": "West", "Hawaii": "West", "Oregon": "West", "Washington": "West",
}

assert len(CENSUS_REGION) == 50
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")


def require_raw_file(filename: str) -> pd.DataFrame:
    path = RAW_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required source file: {path}. Run notebook 01 first.")
    return pd.read_csv(path)


def _normalise_state_string(value: Any) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^A-Za-z ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.upper()


def normalise_state_name(value: Any) -> str | None:
    text = _normalise_state_string(value)
    if text is None:
        return None
    text = text.replace("STATES", "").replace("STATE", "").strip()
    text = re.sub(r"\s+", " ", text)
    if text in NON_STATE_LABELS:
        return None
    if text in EXPECTED_STATE_ABBRS:
        return ABBR_TO_STATE[text]
    # handle title-cased full state names
    title = text.title()
    if title in STATE_TO_ABBR:
        return title
    # fallback for all-uppercase names that include spaces, punctuation, etc.
    if title.lower() in {state.lower() for state in STATES.values()}:
        return next((state for state in STATES.values() if state.lower() == title.lower()), None)
    return None


def to_float(value: Any) -> float | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    token = text.lower().strip()
    if token in {"na", "n/a", "nan", "none", "null", "-", "--", "—", "not available", "not reported"}:
        return None
    text = text.replace(",", "").replace("%", "").replace("$", "")
    text = re.sub(r"[^0-9.\-]", "", text)
    if text in {"", ".", "-", "-.", "+"}:
        return None
    return float(text)


def require_columns(df: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"{name} missing columns: {missing}. Available columns: {list(df.columns)}")


def coerce_numeric_series(
    df: pd.DataFrame,
    column: str,
    source: str,
    *,
    min_value: float | None = None,
    max_value: float | None = None,
) -> pd.Series:
    parsed = df[column].map(to_float)
    na_like = df[column].astype("string").str.strip().str.lower()
    allowed_missing = na_like.isin({"na", "n/a", "nan", "none", "null", "-", "--", "—", "not available", "not reported"}) | na_like.str.contains("insufficient", na=False)
    invalid = parsed.isna() & ~df[column].isna() & ~allowed_missing
    if invalid.any():
        sample = df.loc[invalid, column].head(5).astype("string").tolist()
        raise ValueError(f"{source}.{column} contains non-numeric values: {sample}")
    if min_value is not None:
        below = parsed.notna() & (parsed < min_value)
        if below.any():
            sample = df.loc[below, column].head(5).astype("string").tolist()
            raise ValueError(f"{source}.{column} has values below {min_value}: {sample}")
    if max_value is not None:
        above = parsed.notna() & (parsed > max_value)
        if above.any():
            sample = df.loc[above, column].head(5).astype("string").tolist()
            raise ValueError(f"{source}.{column} has values above {max_value}: {sample}")
    return parsed


def require_50_states(df: pd.DataFrame, state_col: str, source: str) -> pd.DataFrame:
    if state_col not in df.columns:
        raise ValueError(f"{source} missing state column '{state_col}'. Columns: {list(df.columns)}")
    out = df.copy()
    state_input = out[state_col]
    abbr_guess = (
        state_input
        .astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[^A-Z]", "", regex=True)
    )
    from_abbr = abbr_guess.map(ABBR_TO_STATE)
    from_name = state_input.map(normalise_state_name)
    state = from_name.combine_first(from_abbr)
    state_abbr = state.map(STATE_TO_ABBR)
    valid = state_abbr.notna()
    if (~valid).any():
        dropped = int((~valid).sum())
        print(f"{source}: dropping {dropped} non-state rows from '{state_col}'.")
        dropped_samples = state_input.loc[~valid].astype("string").head(10).tolist()
        if dropped <= 10:
            print(f"{source}: dropped rows -> {dropped_samples}")
    out = out.loc[valid].copy()
    out["state_abbr"] = state_abbr.loc[valid]
    out["state"] = state.loc[valid]
    present_states = set(out["state"].dropna())
    extras = sorted(present_states - EXPECTED_STATES)
    missing = sorted(EXPECTED_STATES - present_states)
    if len(present_states) != 50 or missing or extras:
        raise ValueError(
            f"{source} must contain exactly 50 US states after normalisation. "
            f"Found {len(present_states)} states. Missing: {missing}. Unexpected: {extras}."
        )
    if out["state"].duplicated().any():
        dupes = sorted(out.loc[out["state"].duplicated(), "state"].dropna().unique().tolist())
        raise ValueError(f"{source} has duplicated states after normalisation: {dupes}.")
    return out.sort_values("state").reset_index(drop=True)

Raw dir: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\raw
Processed dir: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\processed


## 1. Religiosity

Load the Pew file, range-check the four unaffiliated components (each must be 0–100), and normalise to the 50 states. We then derive the project's headline predictor:

```text
religiously_affiliated_pct = 100 − religiously_unaffiliated_pct
```

Higher `religiously_affiliated_pct` = more religious state. This frame (affiliation, not belief intensity) is the proxy discussed in notebook `01` and the README.

In [3]:
religion = pd.read_csv(RAW_DIR / "pew_religious_unaffiliated_by_state.csv")
require_columns(
    religion,
    ["state", "state_abbr", "religiously_unaffiliated_pct", "atheist_pct", "agnostic_pct", "nothing_in_particular_pct"],
    "religion",
)
for col in [
    "religiously_unaffiliated_pct",
    "atheist_pct",
    "agnostic_pct",
    "nothing_in_particular_pct",
]:
    religion[col] = coerce_numeric_series(religion, col, "religion", min_value=0, max_value=100)
religion = require_50_states(religion, "state", "religion")
religion = religion[[
    "state", "state_abbr", "atheist_pct", "agnostic_pct", "nothing_in_particular_pct", "religiously_unaffiliated_pct"
]].copy()
religion["religiously_affiliated_pct"] = 100.0 - religion["religiously_unaffiliated_pct"]
religion.head()

,state,state_abbr,atheist_pct,agnostic_pct,nothing_in_particular_pct,religiously_unaffiliated_pct,religiously_affiliated_pct
0,Alabama,AL,3.0,2.0,18.0,23.0,77.0
1,Alaska,AK,3.0,9.0,21.0,33.0,67.0
2,Arizona,AZ,3.0,5.0,22.0,30.0,70.0
3,Arkansas,AR,2.0,4.0,12.0,18.0,82.0
4,California,CA,6.0,7.0,20.0,33.0,67.0


## 2. Firearm mortality

A straightforward pass: confirm the expected columns, coerce the rate to a non-negative float, normalise to 50 states, and keep only `state`, `state_abbr`, and `firearm_death_rate_2024`. The raw file is already one row per state, so this mostly enforces types and the 50-state guarantee.

In [4]:
firearm = pd.read_csv(RAW_DIR / "firearm_mortality_by_state_2024.csv")
require_columns(firearm, ["state", "state_abbr", "firearm_death_rate_2024"], "firearm")
firearm["firearm_death_rate_2024"] = coerce_numeric_series(firearm, "firearm_death_rate_2024", "firearm", min_value=0)
firearm = require_50_states(firearm, "state", "firearm")
firearm = firearm[["state", "state_abbr", "firearm_death_rate_2024"]].copy()
firearm.head()

,state,state_abbr,firearm_death_rate_2024
0,Alabama,AL,23.7
1,Alaska,AK,24.4
2,Arizona,AZ,16.9
3,Arkansas,AR,20.6
4,California,CA,7.0


## 3. Obesity

The CDC CSV's column names drift between releases, so instead of hard-coding them this cell **infers** the right columns: it searches for a state-like column (`state`, `location`, …) and a prevalence-like column (`prevalence`, `percent`, `value`, …). It then renames them to `state` / `adult_obesity_pct_2024`, range-checks the percentage, drops the non-state rows the CDC file includes, and enforces the 50-state set.

This inference makes the transform resilient: if CDC renames "Prevalence" to "Percent" next year, the notebook still finds it.

In [5]:
obesity_raw = require_raw_file("adult_obesity_by_state_2024.csv")
print(obesity_raw.columns.tolist())

state_col_candidates = [
    c for c in obesity_raw.columns
    if c.lower() in {"state", "location", "jurisdiction", "geographic area"}
]
if not state_col_candidates:
    state_col_candidates = [
        c for c in obesity_raw.columns
        if c.lower().startswith("state") or "state" in c.lower() or "location" in c.lower()
    ]
if not state_col_candidates:
    raise ValueError("Could not infer state column in obesity CSV.")
state_col = state_col_candidates[0]

prevalence_col_candidates = [
    c for c in obesity_raw.columns
    if any(token in c.lower() for token in ["prevalence", "percent", "percentage", "value", "obesity"])
]
if not prevalence_col_candidates:
    prevalence_col_candidates = [
        c for c in obesity_raw.select_dtypes(include="number").columns if "obesity" not in c.lower()
    ]
if not prevalence_col_candidates:
    raise ValueError("Could not infer obesity prevalence column in obesity CSV.")
prevalence_col = prevalence_col_candidates[0]

obesity = obesity_raw[[state_col, prevalence_col]].copy()
obesity.columns = ["state", "adult_obesity_pct_2024"]
obesity["adult_obesity_pct_2024"] = coerce_numeric_series(
    obesity,
    "adult_obesity_pct_2024",
    "obesity",
    min_value=0,
    max_value=100,
)
# Keep every state even if its prevalence is missing (e.g. CDC "Insufficient data"),
# so the panel still has all 50 states and the gap flows through as NaN.
obesity = obesity.dropna(subset=["state"])
obesity = require_50_states(obesity, "state", "obesity")
obesity = obesity[["state", "state_abbr", "adult_obesity_pct_2024"]]
obesity.head()

['State', 'Prevalence', '95% CI', 'source_url']
obesity: dropping 4 non-state rows from 'state'.
obesity: dropped rows -> ['District of Columbia', 'Guam', 'Puerto Rico', 'Virgin Islands']


,state,state_abbr,adult_obesity_pct_2024
0,Alabama,AL,38.9
1,Alaska,AK,34.0
2,Arizona,AZ,33.3
3,Arkansas,AR,38.9
4,California,CA,29.1


## 4. Imprisonment

Load the BJS file and keep the two 2023 rate columns we care about: `imprisonment_rate_2023_all_ages` (per 100,000 residents) and `imprisonment_rate_2023_adult` (per 100,000 adults). Both are coerced to non-negative numbers and normalised to the 50 states. The all-ages rate is the one used as the headline outcome in notebook `03`; the adult rate is kept as a robustness alternative.

In [6]:
imprisonment = pd.read_csv(RAW_DIR / "bjs_imprisonment_rates_by_state_2023.csv")
require_columns(
    imprisonment,
    ["state", "state_abbr", "imprisonment_rate_2023_all_ages", "imprisonment_rate_2023_adult"],
    "imprisonment",
)
imprisonment["imprisonment_rate_2023_all_ages"] = coerce_numeric_series(
    imprisonment,
    "imprisonment_rate_2023_all_ages",
    "imprisonment",
    min_value=0,
)
imprisonment["imprisonment_rate_2023_adult"] = coerce_numeric_series(
    imprisonment,
    "imprisonment_rate_2023_adult",
    "imprisonment",
    min_value=0,
)
imprisonment = require_50_states(imprisonment, "state", "imprisonment")
imprisonment = imprisonment[[
    "state", "state_abbr", "imprisonment_rate_2023_all_ages", "imprisonment_rate_2023_adult"
]].copy()
imprisonment.head()

,state,state_abbr,imprisonment_rate_2023_all_ages,imprisonment_rate_2023_adult
0,Alabama,AL,394.0,505.0
1,Alaska,AK,202.0,266.0
2,Arizona,AZ,448.0,568.0
3,Arkansas,AR,596.0,773.0
4,California,CA,246.0,314.0


## 5. Literacy and numeracy

The PIAAC file is wide (≈79 columns) with terse names, so this cell selects the columns we need by **pattern-matching on the names** rather than position:

- the state column (`State`),
- an *average literacy score* column — something containing `lit` plus `avg`/`average`/`score` but not `num` → this resolves to `Lit_A`,
- optionally a *low-literacy share* column if one is present.

`Lit_A` is the population average literacy proficiency score (roughly a 0–500 PIAAC scale), so it is range-checked against `0–1000` to allow headroom. The result is normalised to the 50 states. Note the printed column list: if NCES ever changes the schema, that printout is where you would adjust the selectors.

In [7]:
piaac_raw = require_raw_file("piaac_state_literacy_numeracy.csv")
print(piaac_raw.columns.tolist())

state_candidates = [c for c in piaac_raw.columns if c.lower() in {"state", "state_name", "statename", "name"}]
if not state_candidates:
    state_candidates = [c for c in piaac_raw.columns if "state" in c.lower() or "name" in c.lower()]
if not state_candidates:
    raise ValueError("Could not infer PIAAC state column.")
piaac_state_col = state_candidates[0]

columns_lower = {c: c.lower() for c in piaac_raw.columns}
avg_lit_candidates = [
    c for c, lc in columns_lower.items()
    if "lit" in lc and ("avg" in lc or "average" in lc or "score" in lc) and "num" not in lc
]
low_lit_candidates = [
    c for c, lc in columns_lower.items()
    if "lit" in lc and ("low" in lc or "below" in lc or "level_1" in lc or "level1" in lc or "le1" in lc)
]
if not avg_lit_candidates:
    avg_lit_candidates = [c for c, lc in columns_lower.items() if lc in {"lit_a", "literacy_avg_score"}]
    if not avg_lit_candidates:
        raise ValueError("Could not infer average literacy score column. Inspect printed PIAAC columns.")

literacy = piaac_raw[[piaac_state_col, avg_lit_candidates[0]] + low_lit_candidates[:1]].copy()
new_cols = ["state", "literacy_avg_score"]
if low_lit_candidates:
    new_cols.append("low_literacy_pct")
literacy.columns = new_cols
literacy["state"] = literacy["state"].map(normalise_state_name)
literacy = literacy.dropna(subset=["state"])
literacy["literacy_avg_score"] = coerce_numeric_series(
    literacy,
    "literacy_avg_score",
    "literacy",
    min_value=0,
    max_value=1000,
)
if "low_literacy_pct" in literacy.columns:
    literacy["low_literacy_pct"] = coerce_numeric_series(
        literacy,
        "low_literacy_pct",
        "literacy",
        min_value=0,
        max_value=100,
    )
literacy = require_50_states(literacy, "state", "literacy")
literacy = literacy[["state", "state_abbr", "literacy_avg_score"] + (["low_literacy_pct"] if "low_literacy_pct" in literacy.columns else [])]
literacy.head()

['State', 'Lit_P1', 'Lit_P1_CI_L', 'Lit_P1_CI_U', 'Lit_P1_CV', 'Lit_P1_CI_L_indicator', 'Lit_P2', 'Lit_P2_CI_L', 'Lit_P2_CI_U', 'Lit_P2_CV', 'Lit_P3', 'Lit_P3_CI_L', 'Lit_P3_CI_U', 'Lit_P3_CV', 'Lit_P3_indicator', 'Lit_P3_CI_L_indicator', 'Lit_A', 'Lit_A_CI_L', 'Lit_A_CI_U', 'Lit_A_CV', 'Num_P1', 'Num_P1_CI_L', 'Num_P1_CI_U', 'Num_P1_CV', 'Num_P2', 'Num_P2_CI_L', 'Num_P2_CI_U', 'Num_P2_CV', 'Num_P3', 'Num_P3_CI_L', 'Num_P3_CI_U', 'Num_P3_CV', 'Num_P3_indicator', 'Num_P3_CI_L_indicator', 'Num_A', 'Num_A_CI_L', 'Num_A_CI_U', 'Num_A_CV', 'POP', 'POP_TARGET', 'Male', 'Female', 'White', 'Black', 'Hispanic', 'Asian', 'AIAN', 'NHPI', 'Other_race', 'Less_HS', 'HS', 'More_HS', 'Eng_not_well', 'FB_after2010', 'FB_1990_2009', 'FB_before1990', 'FB', 'Poverty_100', 'Poverty_150', 'SNAP', 'Employed', 'Unemployed', 'Not_in_labor', 'OCC_Manage', 'OCC_Service', 'OCC_Sales', 'OCC_Natural', 'OCC_Military', 'OCC_Prod', 'No_Insurance', 'Shape_Length', 'Shape_Area', 'OBJECTID_1', 'STATEFP', 'STUSPS', 'sourc

,state,state_abbr,literacy_avg_score
0,Alabama,AL,258.9
1,Alaska,AK,276.7
2,Arizona,AZ,262.4
3,Arkansas,AR,258.7
4,California,CA,257.2


## 6. Socioeconomic covariates (and numeracy)

State-level religiosity is entangled with poverty, education, the labour market, health-care access, and demographic composition. To move the analysis beyond "region only", we pull a set of **candidate confounders** from the same PIAAC export that gave us literacy (so no extra download is needed):

| Panel column | PIAAC field | Meaning |
|---|---|---|
| `numeracy_avg_score` | `Num_A` | average adult numeracy proficiency (an extra outcome) |
| `poverty_pct` | `Poverty_100` | share below 100% of the poverty line |
| `less_than_hs_pct` | `Less_HS` | share with less than a high-school education |
| `unemployment_pct` | `Unemployed` | unemployment share |
| `uninsured_pct` | `No_Insurance` | share without health insurance |
| `snap_pct` | `SNAP` | share receiving SNAP benefits |
| `pct_black` | `Black` | share Black |
| `pct_hispanic` | `Hispanic` | share Hispanic |

PIAAC stores these as fractions (0–1); we rescale to percentages. Notebook 03 uses them to fit **covariate-adjusted** models and partial correlations — a much stronger test of whether religiosity predicts the outcomes on its own.

In [8]:
piaac_cov_raw = require_raw_file("piaac_state_literacy_numeracy.csv")

# source field -> (panel name, scale factor)
cov_map = {
    "Num_A": ("numeracy_avg_score", 1.0),
    "Poverty_100": ("poverty_pct", 100.0),
    "Less_HS": ("less_than_hs_pct", 100.0),
    "Unemployed": ("unemployment_pct", 100.0),
    "No_Insurance": ("uninsured_pct", 100.0),
    "SNAP": ("snap_pct", 100.0),
    "Black": ("pct_black", 100.0),
    "Hispanic": ("pct_hispanic", 100.0),
}
missing_src = [c for c in cov_map if c not in piaac_cov_raw.columns]
if missing_src:
    raise ValueError(f"PIAAC covariate columns missing from raw file: {missing_src}")

state_candidates = [c for c in piaac_cov_raw.columns if c.lower() in {"state", "state_name", "statename", "name"}]
cov_state_col = state_candidates[0] if state_candidates else "State"

covariates = piaac_cov_raw[[cov_state_col] + list(cov_map)].copy()
covariates = covariates.rename(columns={cov_state_col: "state"})
covariates["state"] = covariates["state"].map(normalise_state_name)
covariates = covariates.dropna(subset=["state"])

for src, (newname, scale) in cov_map.items():
    covariates[newname] = coerce_numeric_series(covariates, src, "covariates", min_value=0) * scale

covariates = require_50_states(covariates, "state", "covariates")
covariates = covariates[["state", "state_abbr"] + [name for name, _ in cov_map.values()]]
covariates.head()


,state,state_abbr,numeracy_avg_score,poverty_pct,less_than_hs_pct,unemployment_pct,uninsured_pct,snap_pct,pct_black,pct_hispanic
0,Alabama,AL,242.5,18.0,14.7,5.2,10.7,15.0,26.5,4.1
1,Alaska,AK,263.7,10.2,7.6,5.8,15.5,10.3,3.2,6.8
2,Arizona,AZ,247.4,17.0,13.5,5.1,12.2,12.5,4.3,30.9
3,Arkansas,AR,242.6,18.1,14.4,4.4,10.6,13.6,15.4,7.2
4,California,CA,243.8,15.1,17.5,5.6,10.5,9.3,5.8,38.8


## 7. Firearm intent, gun-ownership proxy, and population density

Three more pieces, from the new raw files:

- **Firearm homicide & suicide rates** (CDC, 2024) — two outcomes in place of the single total, because they have very different geographies.
- **Gun-ownership proxy** — the FS/S ratio rescaled to a percentage (`gun_ownership_pct_proxy`); the key confounder for firearm outcomes.
- **Population density** — state population (PIAAC `POP`) ÷ land area (Census Gazetteer), an urbanicity proxy.

In [9]:
# --- firearm deaths by intent + gun-ownership proxy (CDC) ---
fa_intent = pd.read_csv(RAW_DIR / "cdc_firearm_intent_by_state_2024.csv")
require_columns(
    fa_intent,
    ["state", "state_abbr", "firearm_homicide_rate_2024", "firearm_suicide_rate_2024", "gun_ownership_fss_proxy"],
    "firearm_intent",
)
for c in ["firearm_homicide_rate_2024", "firearm_suicide_rate_2024"]:
    fa_intent[c] = coerce_numeric_series(fa_intent, c, "firearm_intent", min_value=0)
fa_intent["gun_ownership_pct_proxy"] = coerce_numeric_series(
    fa_intent, "gun_ownership_fss_proxy", "firearm_intent", min_value=0, max_value=1) * 100
fa_intent = require_50_states(fa_intent, "state", "firearm_intent")
fa_intent = fa_intent[["state", "state_abbr", "firearm_homicide_rate_2024", "firearm_suicide_rate_2024", "gun_ownership_pct_proxy"]]

# --- population density (PIAAC population / Census land area) ---
land = pd.read_csv(RAW_DIR / "census_state_land_area.csv")
land["land_area_sqmi"] = coerce_numeric_series(land, "land_area_sqmi", "land", min_value=0)
land = require_50_states(land, "state", "land")[["state", "state_abbr", "land_area_sqmi"]]

piaac_pop = require_raw_file("piaac_state_literacy_numeracy.csv")
pop_state_col = next((c for c in piaac_pop.columns if c.lower() in {"state", "name"}), "State")
pop = piaac_pop[[pop_state_col, "POP"]].copy()
pop.columns = ["state", "population"]
pop["state"] = pop["state"].map(normalise_state_name)
pop = pop.dropna(subset=["state"])
pop["population"] = coerce_numeric_series(pop, "population", "population", min_value=0)
pop = require_50_states(pop, "state", "population")[["state", "state_abbr", "population"]]

geo = land.merge(pop.drop(columns=["state_abbr"]), on="state", validate="one_to_one")
geo["population_density"] = geo["population"] / geo["land_area_sqmi"]
geo = geo[["state", "state_abbr", "population_density"]]

extra = fa_intent.merge(geo.drop(columns=["state_abbr"]), on="state", validate="one_to_one")
extra.head()


,state,state_abbr,firearm_homicide_rate_2024,firearm_suicide_rate_2024,gun_ownership_pct_proxy,population_density
0,Alabama,AL,11.0,12.2,74.313023,95.768828
1,Alaska,AK,4.5,19.2,63.513514,1.293342
2,Arizona,AZ,4.4,12.6,61.543536,59.917491
3,Arkansas,AR,6.9,13.1,67.905405,57.276193
4,California,CA,3.1,3.9,38.040776,250.115887


## 8. Build analytical panel

Now we assemble the panel. Starting from the religiosity table (the seed, guaranteed to have all 50 states), we **left-join** each outcome on `state`. Two safeguards matter here:

- `validate="one_to_one"` makes pandas raise if any join key is duplicated — a guarantee that we never accidentally fan out rows.
- An assertion checks the row count is unchanged (still 50) after every merge.

We then attach `census_region`, order the columns into a readable layout (identifiers → religiosity → outcomes), sort by state, and write the final `state_religiosity_outcomes.csv`. A left join means that if an outcome is missing for a state, that cell becomes `NaN` rather than dropping the state — the missingness is preserved for the analysis notebook to report honestly.

In [10]:
panel = religion.copy()
assert panel.shape[0] == 50, f"Religion seed must have 50 rows, found {panel.shape[0]}."

for df, name in [
    (firearm, "firearm"),
    (obesity, "obesity"),
    (imprisonment, "imprisonment"),
    (literacy, "literacy"),
    (covariates, "covariates"),
    (extra, "extra"),
]:
    before = panel.shape[0]
    panel = panel.merge(df.drop(columns=["state_abbr"], errors="ignore"), on="state", how="left", validate="one_to_one")
    after = panel.shape[0]
    assert before == after, f"Merge with {name} changed row count."

panel = require_50_states(panel, "state", "panel")
panel["census_region"] = panel["state"].map(CENSUS_REGION)

ordered_columns = [
    "state", "state_abbr", "census_region",
    "religiously_affiliated_pct", "religiously_unaffiliated_pct", "atheist_pct", "agnostic_pct", "nothing_in_particular_pct",
    "firearm_death_rate_2024", "firearm_homicide_rate_2024", "firearm_suicide_rate_2024", "adult_obesity_pct_2024",
    "literacy_avg_score", "low_literacy_pct", "numeracy_avg_score",
    "imprisonment_rate_2023_all_ages", "imprisonment_rate_2023_adult",
    "poverty_pct", "less_than_hs_pct", "unemployment_pct", "uninsured_pct", "snap_pct",
    "pct_black", "pct_hispanic", "gun_ownership_pct_proxy", "population_density",
]
ordered_columns = [col for col in ordered_columns if col in panel.columns]
panel = panel[ordered_columns].sort_values("state").reset_index(drop=True)

panel.to_csv(PROCESSED_DIR / "state_religiosity_outcomes.csv", index=False)
panel.head()


,state,state_abbr,census_region,religiously_affiliated_pct,religiously_unaffiliated_pct,atheist_pct,agnostic_pct,nothing_in_particular_pct,firearm_death_rate_2024,firearm_homicide_rate_2024,...,imprisonment_rate_2023_adult,poverty_pct,less_than_hs_pct,unemployment_pct,uninsured_pct,snap_pct,pct_black,pct_hispanic,gun_ownership_pct_proxy,population_density
0,Alabama,AL,South,77.0,23.0,3.0,2.0,18.0,23.7,11.0,...,505.0,18.0,14.7,5.2,10.7,15.0,26.5,4.1,74.313023,95.768828
1,Alaska,AK,West,67.0,33.0,3.0,9.0,21.0,24.4,4.5,...,266.0,10.2,7.6,5.8,15.5,10.3,3.2,6.8,63.513514,1.293342
2,Arizona,AZ,West,70.0,30.0,3.0,5.0,22.0,16.9,4.4,...,568.0,17.0,13.5,5.1,12.2,12.5,4.3,30.9,61.543536,59.917491
3,Arkansas,AR,South,82.0,18.0,2.0,4.0,12.0,20.6,6.9,...,773.0,18.1,14.4,4.4,10.6,13.6,15.4,7.2,67.905405,57.276193
4,California,CA,West,67.0,33.0,6.0,7.0,20.0,7.0,3.1,...,314.0,15.1,17.5,5.6,10.5,9.3,5.8,38.8,38.040776,250.115887


## Data quality checks

A closing audit before the file is trusted downstream. We:

- assert there are no missing values in the **required** analysis columns (so the core predictor and outcomes are complete),
- print the panel shape and a per-column missing-value count,
- confirm 50 unique states and abbreviations,
- print `describe()` so the value ranges can be eyeballed for anything implausible.

In this run the only missing value is **one** state in `adult_obesity_pct_2024` (note the obesity column is not in the strict no-missing list, so a single gap is tolerated and simply reported). Everything else is complete. The panel is now ready for analysis in notebook `03`.

In [11]:
# Required columns must be fully populated. Obesity is deliberately excluded:
# CDC reports "Insufficient data" for some states, so a small gap there is tolerated
# (it is reported below and handled with pairwise dropna in notebook 03).
for col in [
    "religiously_affiliated_pct", "religiously_unaffiliated_pct", "atheist_pct", "agnostic_pct",
    "nothing_in_particular_pct", "firearm_death_rate_2024",
    "literacy_avg_score", "imprisonment_rate_2023_all_ages", "imprisonment_rate_2023_adult"
]:
    if col not in panel.columns:
        continue
    if panel[col].isna().any():
        missing = int(panel[col].isna().sum())
        raise ValueError(f"panel has {missing} missing values in {col}.")
print(panel.shape)
print(panel.isna().sum())
assert panel["state"].nunique() == 50
assert panel["state_abbr"].nunique() == 50
panel.describe(include="all").T

(50, 25)
state                              0
state_abbr                         0
census_region                      0
religiously_affiliated_pct         0
religiously_unaffiliated_pct       0
atheist_pct                        0
agnostic_pct                       0
nothing_in_particular_pct          0
firearm_death_rate_2024            0
firearm_homicide_rate_2024         0
firearm_suicide_rate_2024          0
adult_obesity_pct_2024             1
literacy_avg_score                 0
numeracy_avg_score                 0
imprisonment_rate_2023_all_ages    0
imprisonment_rate_2023_adult       0
poverty_pct                        0
less_than_hs_pct                   0
unemployment_pct                   0
uninsured_pct                      0
snap_pct                           0
pct_black                          0
pct_hispanic                       0
gun_ownership_pct_proxy            0
population_density                 0
dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
state,50,50,Alabama,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_abbr,50,50,AL,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
census_region,50,4,South,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
religiously_affiliated_pct,50.0,NaN,NaN,NaN,70.02,6.819809,52.0,66.25,70.0,74.0,85.0
religiously_unaffiliated_pct,50.0,NaN,NaN,NaN,29.98,6.819809,15.0,26.0,30.0,33.75,48.0
atheist_pct,50.0,NaN,NaN,NaN,4.68,1.984018,1.0,3.0,4.0,5.0,11.0
agnostic_pct,50.0,NaN,NaN,NaN,6.02,2.535382,1.0,5.0,5.0,8.0,12.0
nothing_in_particular_pct,50.0,NaN,NaN,NaN,19.28,4.100971,9.0,17.0,20.0,22.0,29.0
firearm_death_rate_2024,50.0,NaN,NaN,NaN,14.508,5.912451,3.7,11.65,14.05,18.275,28.0
firearm_homicide_rate_2024,50.0,NaN,NaN,NaN,4.464,3.121143,0.8,1.95,3.9,5.8,16.0
